**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/11_t_test_frequentist_vs_bayesian.ipynb)** | **🤖 [AI-Native](11_t_test_frequentist_vs_bayesian.ipynb)** | **🐍 [Python](../python/11_t_test_frequentist_vs_bayesian.ipynb)** | **🔵 [R](../r/11_t_test_frequentist_vs_bayesian.ipynb)** | **⬅️ [Prev: Ch 10](10_mcmc_production_diagnostics.ipynb)** | **➡️ [Next: Ch 12](12_bayesian_regression_and_glms.ipynb)**

*🧭 Part III: Inference, Decision Theory & Case Studies — Chapter 11 of 22*

---

# Chapter 11: Beyond the $p$-Value — Robust Bayesian Two-Group Estimation
### *The Fragility of Welch's $t$-Test, Outliers, and ROPE Decisions*

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> Comparing two groups (A/B testing, canary analysis, benchmark comparisons)
> is the most frequent statistical task in software engineering. When
> developers query an LLM to evaluate experimental results, the AI almost
> always defaults to a classical Student's or Welch's $t$-test. Classical Null
> Hypothesis Significance Testing (NHST) suffers from three fatal structural
> flaws: extreme sensitivity to heavy-tailed outliers (such as garbage
> collection spikes), conflating "absence of evidence" ($p > 0.05$) with
> "evidence of absence", and $p$-hacking through continuous peeking. In this
> chapter, we transition from sampling engines to applied decision-making:
> implementing Kruschke's Bayesian Estimation Supersedes the $t$-Test (BEST)
> with heavy-tailed Student-$t$ likelihoods, Relative Uplift distributions, and
> the Region of Practical Equivalence (ROPE).

## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

When software engineers evaluate an A/B test or optimization experiment using an
LLM, they typically paste raw summary metrics or arrays into the prompt.

### The Naive Prompt
```text
We tested an engine optimization on our backend query pipeline. Group A
(Control, N=60) has mean latency 100.2 ms. Group B (Optimized, N=60) has raw
mean latency 98.4 ms, but includes two garbage collection spikes of 450 ms and
520 ms. Is performance improvement statistically significant? Should we ship?
```

### The Plausible-Sounding AI Hallucination
An unconstrained LLM generates the following confident but disastrous advice:
> *"I ran a two-sample Welch's $t$-test comparing Control vs. Optimized:
> * Control Mean: $\bar{y}_A = 100.2$ ms, Standard Deviation: $s_A = 9.8$ ms
> * Optimized Mean: $\bar{y}_B = 98.4$ ms, Standard Deviation: $s_B = 48.2$ ms

> The calculated $t$-statistic is:
> $$t = \frac{100.2 - 98.4}{\sqrt{\frac{9.8^2}{60} + \frac{48.2^2}{60}}} = \frac{1.80}{\sqrt{1.60 + 38.72}} = \frac{1.80}{6.35} \approx 0.283$$
> With degrees of freedom $\approx 64.9$, the two-tailed $p$-value is:
> $$p = 0.778 \gg 0.05$$
> (Even without variance pooling, $p = 0.283$).

> **Conclusion:** Because $p > 0.05$, there is **no statistically significant
> difference** between Control and Optimized. The observed 1.8 ms difference is
> pure random noise. You should **not** ship this optimization; maintain the
> status quo."*

```
                 THE OUTLIER VARIANCE INFLATION TRAP
                 
   Latency (ms)
     ^
 500 |                                           * (Outlier: 520 ms)
 450 |                                           * (Outlier: 450 ms)
     |
 120 |     * * * * *                           * * * * *
 100 |   * * * * * * * Control (Mean: 100 ms)
  90 |                                       * * * * * * * Optimized (True: 92 ms)
     +-------------------------------------------------------------------->
                  Group A (Control)                   Group B (Optimized)
                  
  [Classical t-Test: 2 Outliers inflate s_B from 9.8 ms to 48.2 ms!]
  [Variance Blowup destroys t-statistic: t drops from 3.2 to 0.28, p explodes!]
  [Naive AI declares 'No difference exists' -- Discarding a true 8 ms win!]
```

---

### Forensic Post-Mortem: Why the AI Failed
The LLM fell into the three classic traps of classical Null Hypothesis
Significance Testing (NHST):

1. **Quadratic Variance Inflation from Outliers:**
   The classical $t$-test assumes that deviations from the group mean follow a
   Gaussian distribution. Under Gaussianity, sample variance sums squared
   residuals:
   $$s^2 = \frac{1}{n-1} \sum_{i=1}^n (y_i - \bar{y})^2$$
   The two GC spikes ($450$ ms and $520$ ms) contributed $(450 - 98)^2 + (520 -
   98)^2 = 123,904 + 178,084 = 301,988$ to the sum of squares, blowing up
   Group B's sample variance by a factor of **$24\times$**! This artificial
   variance balloon inflated the denominator of the $t$-statistic, destroying
   statistical power.

2. **The "Absence of Evidence" Fallacy:**
   The AI claimed: *"Because $p > 0.05$, there is no statistically significant
   difference... maintain status quo."* A high $p$-value **never** proves that
   the null hypothesis is true; it only means the data are noisy under an
   assumed Gaussian model. Conflating $p > 0.05$ with "zero effect" is a fatal
   epistemic error.

3. **The Solution: Heavy-Tailed Student-$t$ Likelihood (BEST):**
   In real production systems, latency data are heavy-tailed. Instead of
   forcing data into a fragile Gaussian mold, Kruschke's Bayesian Estimation
   Supersedes the $t$-Test (BEST) models observations using a Student-$t$
   likelihood:
   $$y_{ij} \sim \text{Student-}t(\nu, \mu_j, \sigma_j)$$
   The normality parameter $\nu$ acts as a self-tuning outlier damper. When
   $\nu \approx 4$, the distribution develops heavy, polynomial tails. The GC
   spikes are naturally accommodated in the fat tails without dragging the mean
   $\mu_B$ or artificially inflating $\sigma_B$.

4. **ROPE vs. Point Null:**
   Real engineering decisions require a **Region of Practical Equivalence
   (ROPE)** (e.g., $[-1.0, +1.0]$ ms). A difference smaller than $1$ ms has zero
   business value. The decision must be based on the posterior probability that
   the improvement exceeds the ROPE threshold:
   $$P(\mu_A - \mu_B > 1.0\text{ ms}) \approx 99.8\%$$
   The optimization was an unambiguous, massive success!

## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Group A (n_A=30) vs Group B (n_B=30) query latency with 2 severe
   GC-pause outliers.
2. Likelihood: Robust Student-t likelihood y ~ Student-t(nu, mu, sigma) with
   normality parameter nu ~ Exp(1/30) to absorb outliers.
3. Decision: Define ROPE = [-2.0, 2.0] ms; report P(Relative Uplift > 5%)
   and P(mu_B - mu_A in ROPE).
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Model:    y_{g,i} ~ Student-t(nu, mu_g, sigma_g),  nu ~ Exp(1/30)
Uplift:   delta = mu_B - mu_A,  ROPE = [-2.0, 2.0] ms
Decision: Ship B if P(delta < -2.0 ms | D) >= 0.95
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as an A/B Testing Statistician. We are comparing query latency between two
variants with occasional latency spikes. Ask me 3 questions about outlier handling,
degrees of freedom, and practical equivalence before running any comparison."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
# ==============================================================================
# PROBABILISTIC SPECIFICATION: ROBUST BAYESIAN TWO-GROUP ESTIMATION (BEST)
# ==============================================================================
module: "ai/11_t_test_frequentist_vs_bayesian"
task: "two_group_latency_comparison"

observables:
  group_a_control: "latencies_control_N60"
  group_b_variant: "latencies_variant_N60"

model_structure:
  likelihood_family: "Student_t"
  parameters:
    - name: "mu_A"
      role: "robust_mean_control"
      prior: "Normal(loc=mean_pooled, scale=3 * std_pooled)"
    - name: "mu_B"
      role: "robust_mean_variant"
      prior: "Normal(loc=mean_pooled, scale=3 * std_pooled)"
    - name: "sigma_A"
      role: "scale_control"
      prior: "Uniform(0.1 * std_pooled, 10 * std_pooled)"
    - name: "sigma_B"
      role: "scale_variant"
      prior: "Uniform(0.1 * std_pooled, 10 * std_pooled)"
    - name: "nu"
      role: "normality_degrees_of_freedom"
      prior: "Exponential(rate=1.0 / 29.0) + 1.0"  # Shifted exponential

derived_quantities:
  mean_difference: "delta_mu = mu_A - mu_B"
  relative_uplift_percent: "uplift = (mu_A - mu_B) / mu_A * 100.0"
  effect_size_cohen_d: "d = (mu_A - mu_B) / sqrt((sigma_A^2 + sigma_B^2) / 2)"

decision_loss_contract:
  rope_interval: "[-1.0, +1.0] ms"
  decision_rules:
    ship_optimization:
      condition: "P(delta_mu > 1.0 ms) >= 0.95 and HDI_95_lower > 1.0 ms"
    maintain_status_quo:
      condition: "HDI_95 completely contained inside [-1.0, +1.0] ms"
    inconclusive_collect_more_data:
      condition: "HDI_95 overlaps ROPE boundary [-1.0, 1.0] ms"
```

</details>


## Part 3: The 4 Mandatory Agentic Verification Gates

Before executing any business or deployment action based on two-group comparison
telemetry, the AI agent must pass four verification gates:

### Gate 1: Normality & Heavy-Tail Diagnostic Gate
* **Invariant:** The degrees-of-freedom parameter $\nu$ must be evaluated. If
  $P(\nu < 30) > 0.5$, the data exhibit significant non-Gaussian heavy tails; a
  classical Gaussian $t$-test is formally invalid.
* **Verification Logic:**
  Assert that Student-$t$ likelihood was employed. Verify that $\nu$ converged
  with $\hat{R} < 1.01$. If empirical $\nu < 10$, explicitly report that heavy
  tails were absorbed and Gaussian assumptions were rejected.

### Gate 2: Relative Uplift Posterior Gate
* **Invariant:** Rather than reporting a point estimate difference, the agent
  must compute the full posterior distribution of Relative Uplift:
  $$\Delta_{\text{rel}} = \frac{\mu_A - \mu_B}{\mu_A} \times 100\%$$
* **Verification Logic:**
  Calculate $P(\Delta_{\text{rel}} > 0\%)$ and the $95\%$ Highest Density
  Interval (HDI) of relative uplift. Ensure that uncertainty in both baseline
  mean $\mu_A$ and treatment mean $\mu_B$ is fully propagated into the ratio.

### Gate 3: ROPE Decision Invariant Gate
* **Invariant:** The decision contract must execute one of three mutually
  exclusive outcomes against the Region of Practical Equivalence (ROPE):
  1. **Ship / Accept Superiority:** The 95% HDI of $\Delta \mu$ falls entirely
     above the upper ROPE bound ($> +1.0$ ms).
  2. **Accept Practical Equivalence:** The 95% HDI falls entirely inside the
     ROPE interval ($[-1.0, +1.0]$ ms).
  3. **Undecided / Continue Experiment:** The 95% HDI overlaps either boundary
     of the ROPE.

### Gate 4: Posterior Predictive Outlier Replication Gate
* **Invariant:** Replicated synthetic datasets $y_{\text{rep}} \sim
  \text{Student-}t(\nu, \mu, \sigma)$ must reproduce extreme observations
  (such as p99 latency spikes) without shifting the central posterior mode.
* **Verification Logic:**
  Simulate $1,000$ replicated datasets. Verify that the 99th percentile of
  replicated data spans the observed outliers ($> 450$ ms), while simulated
  medians match observed group medians ($92$ ms vs $100$ ms).

## Part 4: Worked Agent Execution Transcript & Telemetry

When an epistemic AI agent executes the Bayesian BEST specification on the
two-group latency experiment with outliers, it produces the following verified
execution transcript:

```text
[STEP 1: DATA SUMMARY & RAW COMPARISON]
  Group A (Control): N = 60
    Sample Mean: 100.21 ms, Sample SD: 9.82 ms, Median: 100.10 ms
  Group B (Optimized with 2 Outliers): N = 60
    Raw Sample Mean: 98.40 ms, Raw Sample SD: 48.20 ms, Median: 92.05 ms
    Outlier Points: [450.2 ms, 520.1 ms] (Garbage collection pauses)

[STEP 2: CLASSICAL FREQUENTIST WELCH T-TEST (COMPARISON)]
  t-statistic: 0.283
  Degrees of Freedom: 64.9
  Two-tailed p-value: 0.778
  Frequentist Verdict: FAILED TO REJECT NULL (Discards true 8 ms improvement!)

[STEP 3: BAYESIAN ROBUST STUDENT-T ESTIMATION (BEST)]
  Sampler: 4 Chains, 2000 Draws each (1000 Warmup Discarded = 4000 Draws)
  Convergence: Max R_hat = 1.001 < 1.01 (All chains converged)
  Min ESS: ESS_bulk = 2,890, ESS_tail = 2,640 (PASSED Gates 1 & 2)

  Posterior Estimates:
    Robust Control Mean mu_A: 100.12 ms (95% HDI: [97.55, 102.68])
    Robust Optimized Mean mu_B: 92.14 ms (95% HDI: [89.62, 94.70])
    Normality Parameter nu: 4.28 (95% HDI: [2.15, 8.42])
    Heavy-Tail Diagnosis: nu = 4.28 << 30 => Heavy tails (PASSED Gate 1)

[STEP 4: DERIVED DECISION METRICS]
  Difference of Means: delta_mu = mu_A - mu_B
    Posterior Mean Difference: +7.98 ms  (95% HDI: [+4.41, +11.55] ms)
    Probability of Any Improvement P(delta_mu > 0): 99.98%
    Probability of Meaningful Win P(delta_mu > +1.0 ms): 99.85%

  Relative Uplift: (mu_A - mu_B) / mu_A * 100%
    Posterior Median Uplift: +7.97%  (95% HDI: [+4.35%, +11.58%])

[STEP 5: ROPE DECISION EVALUATION]
  ROPE Interval: [-1.0 ms, +1.0 ms]
  95% HDI: [+4.41 ms, +11.55 ms]
  ROPE Containment:
    P(delta_mu in ROPE): 0.00%
    P(delta_mu < -1.0 ms): 0.00%
    P(delta_mu > +1.0 ms): 99.85%
  Decision Contract: 95% HDI entirely above upper ROPE bound (+4.41 > +1.0).
  Automated Executive Decision: SHIP OPTIMIZATION IMMEDIATELY (WIN CONFIRMED)
```

### ASCII Difference Distribution vs. ROPE Band
```
 Posterior Latency Improvement: delta_mu = mu_Control - mu_Optimized
 
 Density
   |
0.12+                                  * * *
    |                                * : : : *  [95% HDI: +4.41 to +11.55 ms]
0.08+                              * : : : : : *
    |                            * : : : : : : : *
0.04+                          * : : : : : : : : : *
    |     ROPE Zone          * : : : : : : : : : : : *
    |   [ -1.0 , +1.0 ]     * : : : : : : : : : : : : *
0.00+---[=============]---*---------------------*-----------------> delta_mu
       -2.0   0.0   +2.0      +4.0      +6.0      +8.0     +10.0     +12.0
                               ^                   ^
                               |                   +-- Mean Win: +7.98 ms
                               +-- Lower HDI (+4.41 ms) Completely Clears ROPE!
 
  [Welch t-Test: Declared 'No Difference' (p = 0.78) due to 2 Outliers!]
  [BEST: Absorbed Outliers via nu = 4.3; Confirmed 99.85% Win Probability!]
```

### Production Telemetry Table
| Method | Latency Diff $\Delta \mu$ | 95% Interval / HDI | Outlier Handling | $P(\text{Meaningful Win})$ | Executive Action |
|---|---|---|---|---|---|
| **Classical $t$-Test** | $+1.80$ ms | $[-10.9, +14.5]$ ms | Inflates Variance $24\times$ | Undefined ($p=0.78$) | **Cancel Rollout (False Neg)** |
| **Bayesian BEST** | **$+7.98$ ms** | **$[+4.41, +11.55]$ ms** | **Absorbed ($\nu=4.3$)** | **$99.85\%$** | **Ship Optimization (Win)** |

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Dimension | Naive AI Prompt | Calibrated Epistemic Specification |
|---|---|---|
| **Likelihood Model** | Strict Gaussian (violently sensitive to heavy-tailed outliers) | Robust Student-$t$ likelihood with inferable degrees of freedom $\nu$ |
| **Null Interpretation** | High $p$-value misinterpreted as "proven zero effect" | Explicitly quantifies posterior probability of improvement $P(\Delta > 0)$ |
| **Business Impact** | Binary accept/reject; ignores practical engineering thresholds | Incorporates Region of Practical Equivalence (ROPE) |
| **Continuous Peeking** | Vulnerable to false positives under continuous monitoring | Immune to $p$-hacking; posterior updates naturally as data arrives |
| **Outlier Robustness** | 2 GC spikes destroy statistical power and abort rollout | Self-tuning normality parameter $\nu$ absorbs outliers harmlessly |
| **Executive Decision** | "No significant difference; discard patch" (Costly mistake) | "99.85% probability win exceeds 1 ms; deploy to production" |

## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer asks an AI model to evaluate an optimization with $N=500,000$
queries per group:
> *"Control latency mean is $45.002$ ms. Variant latency mean is $45.001$ ms.
> The Welch $t$-test yields $t = 4.82$ with $p = 0.0000014 < 0.001$. The AI
> responds: 'Because the $p$-value is $1.4 \times 10^{-6}$, this performance
> improvement is extraordinarily significant and has an infinitesimal chance of
> being noise. We strongly recommend allocating a $500,000 infrastructure
> migration budget to deploy this optimization immediately.' "*

Explain why the AI's executive recommendation is completely uncalibrated.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Forensic Audit:**
The AI has fallen into the **Large-$N$ Fallacy (Statistical vs. Practical
Significance)**:
1. **Infinitesimal Effect Size:** The actual observed difference is $45.002 -
   45.001 = 0.001$ ms ($1$ microsecond). In a database system, a $1 \mu s$
   difference is imperceptible to users and has zero economic value.
2. **$p$-Values Collapse at Large $N$:** The standard error is $\text{SE} =
   s / \sqrt{N}$. When $N = 500,000$, $\text{SE} \approx 0.0002$ ms. With
   infinite data, even completely trivial differences yield $p < 10^{-6}$.
3. **The ROPE Antidote:** Any realistic engineering contract would set a ROPE
   of $[-0.5, +0.5]$ ms. The entire posterior distribution falls inside the
   ROPE ($P(\Delta \mu \in \text{ROPE}) = 1.000$). The correct decision is
   **Practical Equivalence**: do not spend $\$500,000$ on negligible change.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
Below is an uncalibrated agent prompt designed to evaluate a canary service
deployment:

```text
Compare canary response times (Group B, N=100) vs production response times
(Group A, N=1000). Run a t-test. If p < 0.05, alert the on-call engineer that
the canary is significantly different from production.
```

Rewrite this prompt into a calibrated declarative specification contract that
uses robust Student-$t$ estimation and a ROPE decision rule.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Calibrated Declarative Specification:**
```yaml
module: "calibrated_canary_comparison"
model:
  likelihood: "Student_t(nu, mu_group, sigma_group)"
  priors:
    mu: "Normal(loc=pooled_median, scale=2.5 * pooled_mad)"
    sigma: "HalfNormal(scale=pooled_mad)"
    nu: "Gamma(alpha=2.0, beta=0.1)"  # Favors moderate tails

decision_contract:
  effect_metric: "delta_latency = mu_canary - mu_prod"
  rope_threshold: "[-2.0, +2.0] ms"
  alert_conditions:
    degradation_alert:
      rule: "P(delta_latency > +2.0 ms) > 0.95"
      severity: "PAGE_ONCALL_DEGRADATION"
    certified_safe:
      rule: "95% HDI of delta_latency is fully inside [-2.0, +2.0] ms"
      action: "AUTO_PROMOTE_CANARY"
    inconclusive:
      rule: "HDI overlaps boundary"
      action: "EXTEND_CANARY_WINDOW"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Architect an autonomous continuous canary evaluation agent in pseudo-code that
ingests streaming latency metrics every 5 minutes, fits a Bayesian robust
difference model, evaluates ROPE without $p$-hacking bias from continuous
monitoring, and triggers an automated deployment decision.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Autonomous Contract Architecture:**
```python
def autonomous_continuous_canary_gate(
    streaming_prod_data, streaming_canary_data, rope_bounds=(-2.0, 2.0)
):
  # 1. Ingest streaming metrics and check minimum sample size
  n_canary = len(streaming_canary_data)
  if n_canary < 50:
    return {'action': 'AWAIT_MORE_DATA', 'n_canary': n_canary}

  # 2. Fit Bayesian robust Student-t model (BEST)
  fit = fit_best_model(streaming_prod_data, streaming_canary_data)

  # Check convergence
  if fit.max_rhat >= 1.01 or fit.min_ess < 400:
    return {'action': 'RETRY_SAMPLING_UNCONVERGED', 'rhat': fit.max_rhat}

  # 3. Compute posterior difference: delta = mu_canary - mu_prod
  delta_draws = fit.posterior['mu_canary'] - fit.posterior['mu_prod']
  hdi_lower, hdi_upper = compute_hdi(delta_draws, cred_mass=0.95)
  prob_degradation = np.mean(delta_draws > rope_bounds[1])

  # 4. Immune to p-hacking: Bayesian posteriors update continuously
  if prob_degradation >= 0.95 and hdi_lower > rope_bounds[1]:
    # Canary is definitively slower by more than 2 ms
    return {
        'action': 'ROLLBACK_CANARY_IMMEDIATELY',
        'reason': f'High probability of degradation ({prob_degradation:.3f})',
        'delta_hdi_95': [hdi_lower, hdi_upper],
    }

  if hdi_lower >= rope_bounds[0] and hdi_upper <= rope_bounds[1]:
    # 95% HDI entirely inside ROPE: Practical equivalence achieved
    return {
        'action': 'PROMOTE_CANARY_TO_100_PERCENT',
        'reason': 'Latency is practically equivalent within ROPE tolerance',
        'delta_hdi_95': [hdi_lower, hdi_upper],
    }

  # HDI overlaps ROPE boundary: Inconclusive, continue collecting data
  return {
      'action': 'CONTINUE_MONITORING',
      'reason': 'Uncertainty band still overlaps ROPE threshold',
      'delta_hdi_95': [hdi_lower, hdi_upper],
  }
```
</details>

## Key Takeaways & Course Navigation

### 🎯 Key Takeaways:
1. **$p$-Values Are Fragile:** Two non-Gaussian outliers can blow up sample
   variance quadratically, destroying statistical power and triggering false
   negatives in classical $t$-tests.
2. **Absorb Outliers with Student-$t$:** Modeling data with a heavy-tailed
   Student-$t$ likelihood lets degrees of freedom $\nu$ absorb spikes into the
   tails without distorting the central mean difference.
3. **Absence of Evidence $\neq$ Evidence of Absence:** A high $p$-value never
   proves the null hypothesis. Always compute the posterior distribution of
   effect size and relative uplift.
4. **Make Decisions with ROPE:** Never test against a sharp point null
   ($\Delta = 0$). Evaluate the posterior distribution against a Region of
   Practical Equivalence (ROPE) to separate real wins from large-$N$ noise.

---

**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/11_t_test_frequentist_vs_bayesian.ipynb)** | **🤖 [AI-Native](11_t_test_frequentist_vs_bayesian.ipynb)** | **🐍 [Python](../python/11_t_test_frequentist_vs_bayesian.ipynb)** | **🔵 [R](../r/11_t_test_frequentist_vs_bayesian.ipynb)** | **⬅️ [Prev: Ch 10](10_mcmc_production_diagnostics.ipynb)** | **➡️ [Next: Ch 12](12_bayesian_regression_and_glms.ipynb)**